# Nelson-Siegel AR(1) Projection

Notebook base para proyectar la curva de tasas con un enfoque simple:

1. Estimar factores `level`, `slope` y `curvature` con Nelson-Siegel.
2. Ajustar un AR(1) separado para cada factor.
3. Proyectar esos factores a horizontes de `1M`, `3M`, `6M` y `12M`.
4. Reconstruir la curva futura usando los betas proyectados.

Esta es una primera capa de forecast, útil como escenario base y no como modelo estructural completo.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve().parent.parent
DATA_PATH = PROJECT_ROOT / 'api_js' / 'data' / 'market_rates.csv'
LAMBDA_VALUE = 0.0609
HORIZON_STEPS = {
    '1M': 21,
    '3M': 63,
    '6M': 126,
    '12M': 252,
}

print('Proyecto:', PROJECT_ROOT)
print('Data:', DATA_PATH)

In [ ]:
def nelson_siegel_loadings(tau_years, lam):
    level = np.ones(len(tau_years))
    slope = (1 - np.exp(-lam * tau_years)) / (lam * tau_years)
    curvature = slope - np.exp(-lam * tau_years)
    return np.column_stack([level, slope, curvature])


def fit_ar1(series):
    x = series[:-1]
    y = series[1:]
    x_mean = x.mean()
    y_mean = y.mean()
    numerator = ((x - x_mean) * (y - y_mean)).sum()
    denominator = ((x - x_mean) ** 2).sum()
    phi = 0.0 if abs(denominator) < 1e-12 else numerator / denominator
    intercept = y_mean - phi * x_mean
    residuals = y - intercept - phi * x
    sigma = np.sqrt(np.mean(residuals ** 2))
    return {'intercept': intercept, 'phi': phi, 'sigma': sigma}


def project_ar1(model, last_value, steps):
    path = []
    current = float(last_value)
    for _ in range(steps):
        current = model['intercept'] + model['phi'] * current
        path.append(current)
    return np.array(path)

In [ ]:
columns = ['TPM', 'SPC_03Y', 'SPC_06Y', 'SPC_1Y', 'SPC_2Y', 'SPC_3Y', 'SPC_4Y', 'SPC_5Y', 'SPC_10Y']
months = np.array([1, 3, 6, 12, 24, 36, 48, 60, 120], dtype=float)
tau_years = months / 12.0
X = nelson_siegel_loadings(tau_years, LAMBDA_VALUE)

df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
for col in columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df_clean = df[['Date', *columns]].dropna().reset_index(drop=True)
print('Filas limpias:', len(df_clean))
df_clean.head()

In [ ]:
betas = []
for _, row in df_clean.iterrows():
    y = row[columns].to_numpy(dtype=float)
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    betas.append({
        'Date': row['Date'],
        'level': beta[0],
        'slope': beta[1],
        'curvature': beta[2],
    })

betas = pd.DataFrame(betas).set_index('Date')
betas.tail()

In [ ]:
ar_models = {
    factor: fit_ar1(betas[factor].to_numpy())
    for factor in ['level', 'slope', 'curvature']
}

pd.DataFrame(ar_models).T

In [ ]:
base_date = betas.index[-1]
base_beta = betas.loc[base_date]
projection_rows = []
curve_rows = []

for horizon, steps in HORIZON_STEPS.items():
    projected_level = project_ar1(ar_models['level'], base_beta['level'], steps)
    projected_slope = project_ar1(ar_models['slope'], base_beta['slope'], steps)
    projected_curvature = project_ar1(ar_models['curvature'], base_beta['curvature'], steps)

    projected_beta = np.array([
        projected_level[-1],
        projected_slope[-1],
        projected_curvature[-1],
    ])
    projected_curve = X @ projected_beta

    projection_rows.append({
        'Horizon': horizon,
        'level': projected_beta[0],
        'slope': projected_beta[1],
        'curvature': projected_beta[2],
    })
    for month, rate in zip(months, projected_curve):
        curve_rows.append({
            'Horizon': horizon,
            'MaturityMonths': month,
            'ProjectedRate': rate,
        })

projection_df = pd.DataFrame(projection_rows)
curve_df = pd.DataFrame(curve_rows)
projection_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

betas.plot(ax=axes[0], title='Nelson-Siegel Factors')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Factor')

observed_last = df_clean.iloc[-1][columns].to_numpy(dtype=float)
axes[1].plot(months, observed_last, 'o-', label=f'Observed {base_date.date()}')
for horizon in HORIZON_STEPS:
    horizon_curve = curve_df[curve_df['Horizon'] == horizon]
    axes[1].plot(horizon_curve['MaturityMonths'], horizon_curve['ProjectedRate'], 'o--', label=f'Projected {horizon}')
axes[1].set_title('Projected Yield Curves')
axes[1].set_xlabel('Maturity (months)')
axes[1].set_ylabel('Yield')
axes[1].legend()
plt.tight_layout()